# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Raja-saab/Flyrank1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1 — Freshness and refresh timing

The research reports that the 31–90 day freshness window was a strong freshness band, with a reported 7.88:1 growth-to-decline ratio. It also reports that older content refreshed within 30 days showed a substantial improvement in observed health and impressions.

The label in this project is based on observed search-performance change. A page is treated as declining when its recent performance falls sufficiently relative to the previous period.

**Methodology question:** Because these are observational relationships, does the validation design support an association rather than a causal interpretation? Pages selected for refresh may already differ from other pages before the refresh.

### Finding 2 — Engagement and visibility

The research reports a relationship between stronger engagement/scroll behavior and higher Health Scores. It also reports differences in Health Score between pages with sustained visibility and pages with sporadic visibility.

**Methodology question:** Could existing search visibility, content age, or other page characteristics partly explain this relationship? A grouped or time-aware validation design can provide a stronger check of whether a relationship generalizes beyond the pages used to develop it.

### Overall methodology view

These questions do not reject the research findings. They identify where the observational design should be interpreted carefully. The findings are useful as directional evidence, but they should not automatically be treated as causal effects.

In [18]:
# =========================================================
# SECTION 1 — Research claim audit
# =========================================================

print("RESEARCH CLAIM AUDIT")
print("=" * 50)

findings = [
    "Freshness / refresh timing",
    "Engagement / visibility"
]

questions = [
    "Could refresh selection create pre-existing differences?",
    "Could visibility or content age partly explain engagement relationships?",
    "Would grouped or time-aware validation improve confidence in generalization?"
]

print("\nTwo findings reviewed:")
for i, finding in enumerate(findings, 1):
    print(f"{i}. {finding}")

print("\nMethodology questions:")
for i, question in enumerate(questions, 1):
    print(f"{i}. {question}")

print(
    "\nConclusion: the findings are treated as observed and "
    "directional evidence rather than causal proof."
)

RESEARCH CLAIM AUDIT

Two findings reviewed:
1. Freshness / refresh timing
2. Engagement / visibility

Methodology questions:
1. Could refresh selection create pre-existing differences?
2. Could visibility or content age partly explain engagement relationships?
3. Would grouped or time-aware validation improve confidence in generalization?

Conclusion: the findings are treated as observed and directional evidence rather than causal proof.


## 2. My model under an honest split

I use a client-level holdout so that records belonging to the same client do not appear in both training and test data.

This is an honest validation design for checking whether the model can generalize across clients. The Week-4 baseline and the Week-5 model are evaluated on the same held-out rows and with the same metrics.

The main comparison metric is Precision@50 because the practical use case is prioritizing a small set of pages for review. Average Precision and ROC AUC are also reported as supporting metrics.

In [19]:
# =========================================================
# SECTION 2 — Model under client-level holdout
# =========================================================

import sys
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path("/content/Flyrank1")

FEATURE_PATH = (
    ROOT / "data" / "processed" / "refresh_feature_vector.csv"
)

BASELINE_PATH = (
    ROOT / "data" / "processed" / "baseline_refresh_queue.csv"
)

df = pd.read_csv(FEATURE_PATH)
baseline = pd.read_csv(BASELINE_PATH)

print("Feature rows:", len(df))
print("Baseline rows:", len(baseline))

# ---------------------------------------------------------
# Load repository utilities
# ---------------------------------------------------------

SCRIPTS_DIR = ROOT / "scripts"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from ml_utils import (
    MODEL_NUMERIC_FEATURES,
    MODEL_CATEGORICAL_FEATURES,
    precision_at_k
)

# ---------------------------------------------------------
# Select official model features
# ---------------------------------------------------------

numeric_features = [
    c for c in MODEL_NUMERIC_FEATURES
    if c in df.columns
]

categorical_features = [
    c for c in MODEL_CATEGORICAL_FEATURES
    if c in df.columns
]

print("\nNumeric features:", numeric_features)
print("Categorical features:", categorical_features)

# ---------------------------------------------------------
# Numeric features
# ---------------------------------------------------------

numeric = (
    df[numeric_features]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

# ---------------------------------------------------------
# Categorical features
# ---------------------------------------------------------

categorical = (
    df[categorical_features]
    .fillna("unknown")
    .astype(str)
)

categorical_encoded = pd.get_dummies(
    categorical,
    prefix=categorical_features,
    dtype=float
)

# ---------------------------------------------------------
# Feature matrix
# ---------------------------------------------------------

X = pd.concat(
    [
        numeric.reset_index(drop=True),
        categorical_encoded.reset_index(drop=True)
    ],
    axis=1
)

y = df["is_declining_label"].astype(int)

print("\nX shape:", X.shape)
print("Declining rate:", f"{y.mean():.3%}")

# ---------------------------------------------------------
# Client-level 80/20 split
# ---------------------------------------------------------

clients = (
    df["client_id"]
    .fillna("unknown")
    .astype(str)
)

unique_clients = clients.unique()

rng = np.random.default_rng(42)

shuffled_clients = rng.permutation(
    unique_clients
)

n_test_clients = max(
    1,
    int(round(len(unique_clients) * 0.20))
)

test_clients = set(
    shuffled_clients[:n_test_clients]
)

test_mask = clients.isin(test_clients)

train_idx = np.where(~test_mask)[0]
test_idx = np.where(test_mask)[0]

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# ---------------------------------------------------------
# Check grouping
# ---------------------------------------------------------

train_clients = set(clients.iloc[train_idx])
test_clients = set(clients.iloc[test_idx])

overlap = train_clients & test_clients

print("\nCLIENT-LEVEL SPLIT")
print("=" * 50)
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Training rows:", len(train_idx))
print("Test rows:", len(test_idx))
print("Client overlap:", len(overlap))

assert len(overlap) == 0

print("PASS: no client appears in both partitions.")

# ---------------------------------------------------------
# Train Logistic Regression
# ---------------------------------------------------------

model = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        (
            "logistic",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

model.fit(X_train, y_train)

model_scores = model.predict_proba(X_test)[:, 1]

# ---------------------------------------------------------
# Model metrics
# ---------------------------------------------------------

model_p50 = precision_at_k(
    y_test,
    model_scores,
    50
)

model_ap = average_precision_score(
    y_test,
    model_scores
)

model_auc = roc_auc_score(
    y_test,
    model_scores
)

# ---------------------------------------------------------
# Baseline on EXACT same test rows
# ---------------------------------------------------------

baseline_lookup = (
    baseline
    .drop_duplicates("content_id")
    .set_index("content_id")
    ["baseline_refresh_score"]
)

baseline_scores = (
    df.iloc[test_idx]["content_id"]
    .map(baseline_lookup)
    .fillna(0)
    .to_numpy()
)

baseline_p50 = precision_at_k(
    y_test,
    baseline_scores,
    50
)

baseline_ap = average_precision_score(
    y_test,
    baseline_scores
)

baseline_auc = roc_auc_score(
    y_test,
    baseline_scores
)

# ---------------------------------------------------------
# Comparison
# ---------------------------------------------------------

validation_comparison = pd.DataFrame(
    {
        "Precision@50": [
            baseline_p50,
            model_p50
        ],
        "Average Precision": [
            baseline_ap,
            model_ap
        ],
        "ROC AUC": [
            baseline_auc,
            model_auc
        ]
    },
    index=[
        "Week-4 baseline",
        "Week-5 model"
    ]
)

display(
    validation_comparison.round(3)
)

print(
    f"\nPrecision@50 difference: "
    f"{model_p50 - baseline_p50:+.3f}"
)

Feature rows: 30000
Baseline rows: 30000

Numeric features: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
Categorical features: ['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']

X shape: (30000, 52)
Declining rate: 54.207%

CLIENT-LEVEL SPLIT
Training clients: 26
Test clients: 6
Training rows: 27675
Test rows: 2325
Client overlap: 0
PASS: no client appears in both partitions.


,Precision@50,Average Precision,ROC AUC
Week-4 baseline,0.24,0.468,0.627
Week-5 model,0.40,0.522,0.700



Precision@50 difference: +0.160


## 3. Leakage audit

I checked the final model feature set for known label-derived fields and identifiers.

The target `is_declining_label` must not be used as an input. Outcome-derived fields such as `trend_direction` and `trend_pct` are also excluded because they directly describe the outcome used to construct the label.

`content_id` and `client_id` are identifiers. `client_id` is used for grouped validation but is not used as a predictive feature.

The audit also verifies that the training and test client groups have zero overlap.

This audit checks known leakage risks; it is not a proof that every possible unknown leakage source has been eliminated.

In [20]:
# =========================================================
# SECTION 3 — Leakage audit
# =========================================================

print("LEAKAGE AUDIT")
print("=" * 50)

known_leaky_fields = {
    "is_declining_label",
    "trend_direction",
    "trend_pct"
}

identifier_fields = {
    "content_id",
    "client_id"
}

model_features = set(X.columns)

label_leakage = (
    model_features &
    known_leaky_fields
)

identifier_leakage = (
    model_features &
    identifier_fields
)

print(
    "Known label-derived fields in X:",
    label_leakage
)

print(
    "Identifiers in X:",
    identifier_leakage
)

# Check client separation again
print(
    "\nClient overlap:",
    len(train_clients & test_clients)
)

assert not label_leakage
assert not identifier_leakage
assert len(train_clients & test_clients) == 0

print("\nPASS")
print("- No known label-derived fields are model inputs.")
print("- No identifiers are model inputs.")
print("- Train/test client groups do not overlap.")

LEAKAGE AUDIT
Known label-derived fields in X: set()
Identifiers in X: set()

Client overlap: 0

PASS
- No known label-derived fields are model inputs.
- No identifiers are model inputs.
- Train/test client groups do not overlap.


## 4. Claim rewrite

### Observed result

The model and Week-4 baseline were evaluated on the same client-level holdout. Precision@50 is the primary decision-support metric because the workflow prioritizes a small number of pages for review.

### Safe interpretation

The measured difference between the model and baseline is an observed result on this validation split. If the model has higher Precision@50, the result provides directional evidence that the model may improve prioritization on this dataset. If it has lower Precision@50, the baseline remains stronger under this evaluation.

The result does not establish causality and does not guarantee the same performance on future clients or future data.

In [21]:
# =========================================================
# SECTION 4 — Error analysis
# =========================================================

# ---------------------------------------------------------
# Create test-result table
# ---------------------------------------------------------

error_df = df.iloc[test_idx][
    [
        "content_id",
        "client_id",
        "is_declining_label"
    ]
].copy()

error_df["actual"] = y_test.to_numpy()
error_df["model_score"] = model_scores
error_df["baseline_score"] = baseline_scores

# ---------------------------------------------------------
# Ranking positions
# ---------------------------------------------------------

error_df["model_rank"] = (
    error_df["model_score"]
    .rank(
        ascending=False,
        method="first"
    )
)

error_df["baseline_rank"] = (
    error_df["baseline_score"]
    .rank(
        ascending=False,
        method="first"
    )
)

# ---------------------------------------------------------
# Top-50 model errors
# ---------------------------------------------------------

model_top50 = (
    error_df
    .sort_values(
        "model_score",
        ascending=False
    )
    .head(50)
)

false_positives = (
    model_top50["actual"] == 0
).sum()

true_positives = (
    model_top50["actual"] == 1
).sum()

print("MODEL TOP-50 ERROR ANALYSIS")
print("=" * 50)

print(
    "True positives in top 50:",
    true_positives
)

print(
    "False positives in top 50:",
    false_positives
)

print(
    "Measured Precision@50:",
    f"{model_p50:.3f}"
)

# ---------------------------------------------------------
# Compare errors between model and baseline
# ---------------------------------------------------------

baseline_top50 = (
    error_df
    .sort_values(
        "baseline_score",
        ascending=False
    )
    .head(50)
)

baseline_tp = (
    baseline_top50["actual"] == 1
).sum()

baseline_fp = (
    baseline_top50["actual"] == 0
).sum()

print("\nBASELINE TOP-50")
print("=" * 50)

print(
    "True positives:",
    baseline_tp
)

print(
    "False positives:",
    baseline_fp
)

print(
    "Measured Precision@50:",
    f"{baseline_p50:.3f}"
)

# ---------------------------------------------------------
# Final interpretation
# ---------------------------------------------------------

print("\nINTERPRETATION")
print("=" * 50)

if model_p50 > baseline_p50:

    print(
        "The model measured higher Precision@50 than "
        "the baseline on this holdout."
    )

elif model_p50 < baseline_p50:

    print(
        "The baseline measured higher Precision@50 than "
        "the model on this holdout."
    )

else:

    print(
        "The model and baseline measured the same "
        "Precision@50 on this holdout."
    )

print(
    "\nThe errors are ranking errors: some pages ranked "
    "highly by the model are not actually in the declining "
    "class. These false positives limit Precision@50."
)

print(
    "This analysis is decision-support evidence rather "
    "than a causal claim about why individual pages decline."
)

MODEL TOP-50 ERROR ANALYSIS
True positives in top 50: 20
False positives in top 50: 30
Measured Precision@50: 0.400

BASELINE TOP-50
True positives: 12
False positives: 38
Measured Precision@50: 0.240

INTERPRETATION
The model measured higher Precision@50 than the baseline on this holdout.

The errors are ranking errors: some pages ranked highly by the model are not actually in the declining class. These false positives limit Precision@50.
This analysis is decision-support evidence rather than a causal claim about why individual pages decline.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.